### Install new libraries

In [84]:
#!pip install ddgs trafilatura

### Setup

In [85]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
from IPython.display import Markdown, display

from ddgs import DDGS
import trafilatura #package design to scrape things from the web. Similar to beautiful soup.

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")

client = OpenAI()

MODEL = "gpt-4.1-mini"

### Step 1: Define the Tools

In [86]:
def search_web(query: str):
    """Search the web using DuckDuckGo browser. Returns 3 results."""
    ddgs = DDGS() #Browser compared to browser Duck Duck Go
    results = ddgs.text(query, max_results=3)
    print(f" \u2705 Got results")
    return json.dumps(results, indent=2)

In [87]:
def fetch_url(url: str):
    """Fetch the content of a UDL using trafilatura."""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f". \u2705 Got text: {len(text)} chars")
            return text
    print(f". \u274C Failed to fetch or extract text.")
    return f"Could not extract text from {url}. Try a different source."

In [88]:
search_web("AI in healthcare in 2030")

 ✅ Got results


'[\n  {\n    "title": "Pulse Report: Healthcare in 2030: How AI Agents Promote ...",\n    "href": "https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/",\n    "body": "With the rapid democratization of AI, the future of healthcare was already under redevelopment. Now, in an era of AI agents, that future is getting reshaped. What might this new healthcare future, powered by AI agents, look like?"\n  },\n  {\n    "title": "AI and Healthcare in 2030: Predictions and Pathways | Journal ...",\n    "href": "https://japmi.org/index.php/japmi/article/view/23",\n    "body": "This paper explores the key advancements expected in AI-driven healthcare, including precision medicine, predictive analytics, and automated workflows, and the challenges posed by ethical considerations, data security, and regulatory frameworks."\n  },\n  {\n    "title": "Artificial intelligence in healthcare and medicine: clinical ...",\n    "hre

In [89]:
fetch_url("https://en.wikipedia.org/wiki/Artificial_intelligence_in_healthcare")

. ✅ Got text: 98592 chars


'Artificial intelligence in healthcare\nArtificial intelligence in healthcare refers to the application of artificial intelligence (AI) to medical and healthcare data in areas including disease diagnosis, treatment planning,[1] patient monitoring, drug development,[2] and clinical decision support systems.[3][4][5]\nThe use of AI in healthcare has also raised ethical, technical, and regulatory concerns, including related to issues such as data privacy, automation of jobs, and amplifying already existing algorithmic bias.[6] Studies have also examined concerns among patients, health professionals, and the public about trust and empathy in care involving AI.[7]\nApplications in healthcare systems\n[edit]\nDisease diagnosis\n[edit]\nAccurate and early diagnosis of diseases is still a challenge in healthcare. Recognizing medical conditions and their symptoms is a complex problem. AI can assist clinicians with data processing capabilities to save time and improve accuracy.[8][9] Through the

### Step 2: Describe as LLM Tools

In [90]:
tools = []

In [91]:
search_web_function = {
    "name": "search_web",
    "description": "Search the web using DuckDuckGo browser. Return 3 results.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to find relevant websites."
            }
        },
        "required": ["query"]
    }
}

tools.append({"type": "function", "function": search_web_function})

In [92]:
#Describe fetch_url as an LLM tool
fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch and extract the main text content from a web page.",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL of the web page to fetch and extract text from."
            }
        },
        "required": ["url"]
    }
}

tools.append({"type": "function", "function": fetch_url_function})

In [93]:
#Check list of tools
#tools

### Step 3: Tool call handler

In [94]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f" \U0001f527 Calling function {function_name} with arguments: {args}") #For future debugging

        #Route to the appropriate function based on function_name
        if function_name == "search_web":
            result = search_web(args["query"])
            content = f"Search results: {result}"
        elif function_name == "fetch_url":
            result = fetch_url(args["url"])
            content = f"Fetched URL content: {result}"
        #elif function_name == "insert_function_name_3":
            # content = insert_function_name3(args["message"])
        #....
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id,
        }

        tool_results.append(tool_call_result)
    
    return tool_results

### Step 4: The System Prompt
This tells the LLM who it is and how to behave. The key things:
 - What its job is
 - What tools it has
 - What process to follow
 - What output format to produce

In [95]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather information from at least 6 distinct sources before delivering your brief.
If you have fewer tthan 6, keep searching.

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that "DONE:" should be at the start of the final response, so that it can be easily parsed and extracted.
You CANNOT and SHOULD NOT include "DONE:" in any other part of your response except at the very BEGINNING of the final research brief.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

In [96]:
RESEARCH_AGENT_PROMPT_2 = """

Your job is to provide information from the web and share it with the user as text. To achieve that you have 
two tools:

- search_web: Search the web using DuckDuckGo browser. Return 3 results.

- fetch_url: Fetch and extract the main text content from a web page.

When the user asks you to research a topic use the search_web tool to browse the topic through the web.
Retrieve the result urls, use them to run the fetch_url tool to retrieve text from each of the results. 
Provide the user with a text description of the obtained information. Separate the summaries for each of 
the results and below each include the corresponding url as a reference.

"""

### Step 5: The Agentic Loop

In [97]:
def run_research_agent(topic: str, max_iterations: int = 10) -> str:
    """
    Run the research agent on a topic and return the research brief.

    Args:
        topic: The topic of research
        max_iterations: Safety limit to prevent infinite loops

    Returns:
        Te research brief as a string
    """
    print(f"\n\U0001F50D Starting research on: {topic}")
    print("=" * 60)

    #Initialize conversation messages list with system prompt + research task
    messages = [
        {"role": "system", "content": RESEARCH_AGENT_PROMPT},
        {"role": "user", "content": f"Research the following topic and produce a comprehensive research brief:\n {topic}"}
    ]

    #LOOP
    iteration = 0
    while iteration < max_iterations:
        iteration += 1
        print(f"\n--- Iteration {iteration} ---")

        #1. Call the LLM and get response
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        message = response.choices[0].message
        print("MESSAGE----------------------------", message)
        print("MESSAGE content----------------------------", message.content)
        messages.append(message)
        #messages.append({"role": "assistant", "content": message.content})

        #2. Check if the LLM called tools
        if message.tool_calls:
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)

        #3. Otherwise: no tools were called, read message content
        else:
            content = message.content #No tool calls, so there's just a normal text response from the LLM
            # Check if DONE:, then return
            if content.startswith("DONE:"):
                research_brief = content[len("DONE:")].strip()
                print(f"\n\u2705 Research complete!")
                return research_brief

            # Otherwise: not yet done, append message
            else:
                print(f"  \U0001F4AD Agent is thinking:")
                pprint(content)
                #Loop continues to next iteration

        #4. If we're entering the final iteration, force a final answer
        if (iteration == max_iterations -1):
            print(f"  \u26a0 Safety limit reached. Stopping research in next iteration.")
            messages.append({"role": "user", "content": "You have reached the maximum number of iterations. "
            "Please deliver your research brief now. You MUST respond with DONE: followed by your brief."})

    #Fallback return
    return "Research incomplete. Maximum iterations reached without finalizing brief."

### Lets Run It!

In [98]:
brief = run_research_agent("AI in healthcare in 2030")
display(Markdown(brief))


🔍 Starting research on: AI in healthcare in 2030

--- Iteration 1 ---
MESSAGE---------------------------- ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_GAIZJ5bOmHvxbSvNHyeKqGO8', function=Function(arguments='{"query":"AI in healthcare 2030"}', name='search_web'), type='function')])
MESSAGE content---------------------------- None
 🔧 Calling function search_web with arguments: {'query': 'AI in healthcare 2030'}
 ✅ Got results

--- Iteration 2 ---
MESSAGE---------------------------- ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Qk9cA9p5yvH8jonMbKXqoxaN', function=Function(arguments='{"url": "https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/"}', name='fet

In [99]:
MODEL = "gpt-4o"
brief = run_research_agent("All the different species of Rhinoceros")
display(Markdown(brief))


🔍 Starting research on: All the different species of Rhinoceros

--- Iteration 1 ---
MESSAGE---------------------------- ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_t3wq8UuJqNNIzgap8vjcVUqi', function=Function(arguments='{"query":"species of Rhinoceros"}', name='search_web'), type='function')])
MESSAGE content---------------------------- None
 🔧 Calling function search_web with arguments: {'query': 'species of Rhinoceros'}
 ✅ Got results

--- Iteration 2 ---
MESSAGE---------------------------- ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_PIn1h53O0VFFd6p016KKAR3y', function=Function(arguments='{"url": "https://en.wikipedia.org/wiki/Rhinoceros"}', name='fetch_url'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_

In [100]:
MODEL = "gpt-4.1-mini"
brief = run_research_agent("All the different species of Rhinoceros")
display(Markdown(brief))


🔍 Starting research on: All the different species of Rhinoceros

--- Iteration 1 ---
MESSAGE---------------------------- ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_4w2DQUFvTqlhV0Hxw9QfgRaU', function=Function(arguments='{"query":"all different species of rhinoceros"}', name='search_web'), type='function')])
MESSAGE content---------------------------- None
 🔧 Calling function search_web with arguments: {'query': 'all different species of rhinoceros'}
 ✅ Got results

--- Iteration 2 ---
MESSAGE---------------------------- ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_fRX30cZQkC1rngYBvpRaxNOR', function=Function(arguments='{"url": "https://en.wikipedia.org/wiki/Rhinoceros"}', name='fetch_url'), type='function'), ChatCompletionMessa

### Evals

In [101]:
JUDGE_PROMPT = """
Judge Prompt: Research Source Breadth (TRUE/FALSE)
You are scoring a research agent’s “research brief” to determine if there was a “source breadth failure”.

Return only "TRUE" or "FALSE".

Definitions
“Source”: A distinct external source referenced or relied upon in the research brief. Examples include a specific webpage, article, report, study, publication, dataset, government page, company page, or other distinct source.

“Source breadth failure (label TRUE): Any of the below occurred:

Fewer than 6 distinct sources referenced: The research brief references or relies upon 5 or fewer distinct external sources.
Repeated references do not count separately: Multiple mentions, links, citations, or references to the same underlying source count as only one source.
Different pages from the same source do not automatically count separately: Multiple pages/articles from the same publication, website, organization, or source should only count as distinct sources when they are clearly separate pieces of research/content. Do not inflate the count based merely on different URLs.
Source breadth cannot be established: If the brief does not provide enough identifiable information to determine that at least 6 distinct external sources were referenced, label TRUE.
“No source breadth failure (label FALSE): The research brief references at least 6 clearly distinct external sources, regardless of whether the sources are formally cited, consistently formatted, or cited in a particular citation style.

Important: Do not evaluate citation correctness, citation formatting, source quality, or whether the sources are authoritative. The only question is whether the deliverable demonstrates breadth by referencing at least 6 distinct sources.

Output Format
Return exactly one token: TRUE or FALSE. No explanations.
"""

In [102]:
TOPICS = [
    "How has CRISPR gene editing evolved and what are its current applications?",
    "What factors have driven the growth of the global electric vehicle market since 2020?",
    "What were the major causes and consequences of the collapse of the Soviet Union?",
    "What strategies can cities use to reduce urban heat and adapt to rising temperatures?",
    "How effective is telemedicine for managing chronic diseases?",
    "How has generative AI changed higher education, teaching, assessment, and academic integrity?",
    "How have streaming services changed the economics of film and television?",
    "What are the major challenges and opportunities in expanding renewable energy grids?",
    "What factors most influence performance and recovery in elite endurance athletes?",
    "How can precision agriculture improve food production while reducing environmental impact?",
]

In [ ]:
results = []

for topic in TOPICS:
    brief = run_research_agent(topic, max_iterations=10)

    response = client.responses.create(
        model="gpt-4.1",
        instructions=JUDGE_PROMPT,
        input=f"Research brief:\n\n{brief}",
    )

    verdict = response.output_text.strip()
    results.append({"topic": topic, "verdict": verdict, "brief": brief})

    print(f"{verdict:5} | {topic}")

print("\nSummary:")
print(f"Passed:  {sum(r['verdict'] == 'FALSE' for r in results)}")
print(f"Failed:  {sum(r['verdict'] == 'TRUE' for r in results)}")